<a id="Route-Generator"></a>
<h3 style="color:#1f77b4;">Route Generator</h3>
<h5 style="color:#1fb4a7;">Meter reading route builder using WA GNAF address data</h5>

<a id="Config"></a>
<h3 style="color:#1f77b4;">Config</h3>

In [ ]:
import random
import pandas as pd


base_path = r"C:\Users\thoma\Documents\WA address GNAF"
output_path = r"C:\Users\thoma\Documents\balga_r_deadwalk.csv"


# (street, suburb, property_type)
streets_input = [
    ("STEDHAM", "BALGA", "R"),
    ("FITZROY", "BALGA", "R"),
    ("BREDE", "BALGA", "R"),
    ("ST KILDA", "BALGA", "R"),
    ("GRINSTEAD", "BALGA", "R"),
    ("MENTONE", "BALGA", "R"),

]


# # (street, suburb, property_type)
# streets_input = [
#     ("HILLIER", "HAMERSLEY", "R"),
#     ("DON", "HAMERSLEY", "R"),
#     ("FARMAN", "HAMERSLEY", "R"),
#     ("AVRO", "HAMERSLEY", "R"),
#     ("VICKERS", "HAMERSLEY", "R"),
# ]

seed = 42

<a id="Data-Acquisition"></a>
<h5 style="color:#1fb4a7;">Loading GNAF datasets</h5>

Converted to parquet seperatley for faster processing in python

In [ ]:
address = pd.read_parquet(base_path + "\\WA_ADDRESS_DETAIL_psv.parquet")
geo = pd.read_parquet(base_path + "\\WA_ADDRESS_DEFAULT_GEOCODE_psv.parquet")
streets = pd.read_parquet(base_path + "\\WA_STREET_LOCALITY_psv.parquet")
locality = pd.read_parquet(base_path + "\\WA_LOCALITY_psv.parquet")

address.head()

<a id="Trim-Columns"></a>
<h5 style="color:#1fb4a7;">Trim Columns</h5>
Trim to only needed columns

In [ ]:
streets = streets[["STREET_LOCALITY_PID", "STREET_NAME", "STREET_TYPE_CODE", "LOCALITY_PID"]].rename(
    columns={"LOCALITY_PID": "STREET_LOCALITY_LOCALITY_PID"}
)

locality = locality[["LOCALITY_PID", "LOCALITY_NAME", "PRIMARY_POSTCODE"]]

<a id="Merge"></a>
<h5 style="color:#1fb4a7;">Merge</h5>

Bringing the required fields from each .psv into one dataframe

In [ ]:
df = (
    address
    .merge(geo, on="ADDRESS_DETAIL_PID", how="inner")
    .merge(streets, on="STREET_LOCALITY_PID", how="left")
    .merge(locality, left_on="STREET_LOCALITY_LOCALITY_PID", right_on="LOCALITY_PID", how="left")
)

print(f"{len(df):,} rows after merge")

df.columns

<a id="Clean"></a>
<h5 style="color:#1fb4a7;">Clean</h5>

Renaming columns to keep it clean, making more readable.

In [ ]:
df = df.dropna(subset=["NUMBER_FIRST", "STREET_NAME", "LOCALITY_NAME", "POSTCODE", "LATITUDE", "LONGITUDE"]).copy()

df = df.rename(columns={
    "FLAT_NUMBER": "unit",
    "NUMBER_FIRST": "number",
    "STREET_NAME": "street",
    "STREET_TYPE_CODE": "type",
    "LOCALITY_NAME": "suburb",
    "POSTCODE": "postcode",
    "LATITUDE": "lat",
    "LONGITUDE": "lon"
})

df["number"] = df["number"].astype(int)
df["unit"] = df["unit"].astype("Int64")

df = df[["unit", "number", "street", "type", "suburb", "postcode", "lat", "lon"]].copy()

for col in ["street", "type", "suburb"]:
    df[col] = df[col].astype(str).str.upper().str.strip()

print(f"{len(df):,} rows after cleaning")

df